Set the path to the git repo:

In [2]:
REPO_PATH = "/home/akash/Projects/Point-Policy"
PKL_PATH = "/home/akash/Projects/data/expert_demos/franka_env/bowl.pkl"

## 📋 Instructions

1. **📁 Upload your .pkl file** using the file upload widget 
   - **IF IT'S NOT LETTING YOU COPY/PASTE WITH CTRL C/V USE THE EDIT DROPDOWN IN VSCODE**
   - Usually this is in `.../data/expert_demos/franka_env/task_name.pkl`
   - Task name will be automatically extracted from .pkl filename
   - Internally, images will be loaded from: `data['observations'][0]['pixels1'/'pixels2'][0]`
2. **🎥 Select camera view** (pixels1 or pixels2) and click 'Switch View'
3. **🎯 Enter object name** (e.g., 'lemon', 'cube') and click 'Set Object'
4. **🖱️ Choose annotation mode:**
   - **point**: Click on image to add points
   - **bbox**: Click and drag to create bounding boxes
5. **💾 Click 'Save Object'** to save current object's annotations
6. **🔄 Switch between camera views** and repeat for same objects
7. **👀 Click 'Show Annotations'** to view current progress
8. **📤 Click 'Save All'** to export JSON and YAML files

### 🖱️ Mouse Controls
- **Point mode**: Left click to add numbered points
- **Bbox mode**: Click and drag to create bounding box

### 🎨 Visual Cues
- **🔴 Current work**: Red points, blue bounding boxes
- **🟢 Saved annotations**: Green points/boxes with labels
- **📦 Bbox preview**: Dashed red rectangle while dragging

### 💾 Output Files
- in a task directory `/coordinates/{task_name}`
   - images for each camera view
   - `annotations.json` file with annotation data
- `{task_name}.yaml` file in `point_policy/cfgs/suite/task/franka_env/`

In [3]:
import json
import yaml
import os
import pickle
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from ipycanvas import Canvas
from IPython.display import display, clear_output
import cv2
from PIL import Image

PIXEL_KEYS = ["pixels1", "pixels2"]

class ImageAnnotationTool:
    def __init__(self):
        self.task_name = None
        self.task_dir = None
        self.annotations = None
        self.current_pixel_key = PIXEL_KEYS[0]
        self.current_object_name = None
        self.data = None
        self.current_image = None
        self.canvas = None
        self.points = []
        self.bbox_start = None
        self.annotation_mode = "point"  # "point" or "bbox"
        self.total_points = 0
        self.is_drawing_bbox = False
        self.canvas_width = 600
        self.canvas_height = 450
        self.image_scale = 1.0
        self.image_offset_x = 0
        self.image_offset_y = 0
        self.cached_resized_image = None
    
        # Auto-load pickle file from PKL_PATH
        self.auto_load_pickle()

    def auto_load_pickle(self):
        """Automatically load pickle file from PKL_PATH"""
        try:
            with open(PKL_PATH, 'rb') as f:
                self.data = pickle.load(f)
            
            filename = Path(PKL_PATH).name
            self.task_name = Path(filename).stem
            self.task_dir = Path(REPO_PATH) / "coordinates" / self.task_name
            
            # Initialize annotations structure
            self.annotations = {
                "task_name": self.task_name,
                "pixel_keys": {}
            }
            
            for pixel_key in PIXEL_KEYS:
                if pixel_key not in self.annotations["pixel_keys"]:
                    self.annotations["pixel_keys"][pixel_key] = {
                        "image_path": f"{pixel_key}.jpg",
                        "objects": []
                    }
            
            self.switch_pixel_key(self.current_pixel_key)
            print(f"✅ Auto-loaded: {filename}")
            print(f"📝 Task name: {self.task_name}")
            
        except Exception as e:
            print(f"❌ Error auto-loading from PKL_PATH: {e}")
            self.task_name = "Error loading file"

    def switch_pixel_key(self, pixel_key):
        """Switch between pixels1 and pixels2"""
        if self.data is None:
            print("No data loaded. Please load a pickle file first.")
            return
            
        self.current_pixel_key = pixel_key
        
        try:
            # Extract image from data structure: data['observations'][0][pixel_key][0]
            img_bgr = self.data['observations'][0][pixel_key][0]
            
            # Convert BGR to RGB for display
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            self.current_image = img_rgb
            
            self.setup_canvas()
            print(f"🔄 Switched to {pixel_key}")
            
        except Exception as e:
            print(f"❌ Error loading image for {pixel_key}: {e}")
            print("Expected data structure: data['observations'][0][pixel_key][0]")
        
    def setup_canvas(self):
        """Setup the canvas for annotation"""
        # Only create canvas if it doesn't exist (will be created in create_ui)
        if self.canvas is None:
            return
        
        # Clear canvas
        self.canvas.clear()
        
        if self.current_image is not None:
            # Calculate scaling to fit image in canvas
            img_height, img_width = self.current_image.shape[:2]
            scale_x = self.canvas_width / img_width
            scale_y = self.canvas_height / img_height
            self.image_scale = min(scale_x, scale_y)
            
            # Calculate offset to center image
            scaled_width = int(img_width * self.image_scale)
            scaled_height = int(img_height * self.image_scale)
            self.image_offset_x = (self.canvas_width - scaled_width) // 2
            self.image_offset_y = (self.canvas_height - scaled_height) // 2
            
            # Resize image and cache it
            self.cached_resized_image = cv2.resize(self.current_image, (scaled_width, scaled_height), interpolation=cv2.INTER_LANCZOS4)
            self.canvas.put_image_data(self.cached_resized_image, self.image_offset_x, self.image_offset_y)
            
            # Draw existing annotations
            self.draw_existing_annotations()
        
        self.update_canvas_title()

    def redraw_canvas_efficiently(self):
        """Efficiently redraw canvas without reloading image"""
        # Clear canvas
        self.canvas.clear()
        
        # Redraw cached image
        if self.cached_resized_image is not None:
            self.canvas.put_image_data(self.cached_resized_image, self.image_offset_x, self.image_offset_y)
        
        # Redraw existing annotations
        self.draw_existing_annotations()
        
        # Redraw current points
        self.draw_current_points()
        
    def update_canvas_title(self):
        """Update the title above the canvas"""
        title = f"Task: {self.task_name} | View: {self.current_pixel_key}"
        if self.current_object_name:
            title += f" | Object: {self.current_object_name}"
        else:
            title += " | No object selected"
        
        return title
        
    def image_to_canvas_coords(self, x, y):
        """Convert image coordinates to canvas coordinates"""
        canvas_x = x * self.image_scale + self.image_offset_x
        canvas_y = y * self.image_scale + self.image_offset_y
        return canvas_x, canvas_y
    
    def canvas_to_image_coords(self, x, y):
        """Convert canvas coordinates to image coordinates"""
        image_x = (x - self.image_offset_x) / self.image_scale
        image_y = (y - self.image_offset_y) / self.image_scale
        return int(image_x), int(image_y)
    
    def is_point_in_image(self, x, y):
        """Check if canvas coordinates are within the image bounds"""
        img_x, img_y = self.canvas_to_image_coords(x, y)
        if self.current_image is None:
            return False
        img_height, img_width = self.current_image.shape[:2]
        return 0 <= img_x < img_width and 0 <= img_y < img_height
        
    def draw_existing_annotations(self):
        """Draw existing annotations on the canvas"""
        if (self.annotations is None or 
            self.current_pixel_key not in self.annotations["pixel_keys"]):
            return
            
        objects = self.annotations["pixel_keys"][self.current_pixel_key]["objects"]
        
        for obj in objects:
            object_name = obj["name"]
            
            # Draw points
            if "points" in obj:
                for i, point in enumerate(obj["points"]):
                    x, y = point["x"], point["y"]
                    canvas_x, canvas_y = self.image_to_canvas_coords(x, y)
                    
                    # Draw green circle for saved points
                    self.canvas.fill_style = "green"
                    self.canvas.begin_path()
                    self.canvas.arc(canvas_x, canvas_y, 4, 0, 2 * np.pi)
                    self.canvas.fill()
                    
                    # Draw label
                    self.canvas.fill_style = "green"
                    self.canvas.font = "12px Arial"
                    self.canvas.fill_text(f"{object_name}_{i+1}", canvas_x + 6, canvas_y - 6)
            
            # Draw bounding box
            if "bounding_box" in obj:
                bbox = obj["bounding_box"]
                x1, y1 = self.image_to_canvas_coords(bbox["xmin"], bbox["ymin"])
                x2, y2 = self.image_to_canvas_coords(bbox["xmax"], bbox["ymax"])
                
                # Draw green rectangle for saved bbox
                self.canvas.stroke_style = "green"
                self.canvas.line_width = 2
                self.canvas.stroke_rect(x1, y1, x2 - x1, y2 - y1)
                
                # Draw label
                self.canvas.fill_style = "green"
                self.canvas.font = "14px Arial"
                self.canvas.fill_text(object_name, x1, y1 - 5)
        
    def on_mouse_down(self, x, y):
        """Handle mouse down events"""
        if not self.is_point_in_image(x, y) or self.current_object_name is None:
            return
            
        if self.annotation_mode == "point":
            self.add_point(x, y)
        elif self.annotation_mode == "bbox":
            self.bbox_start = (x, y)
            self.is_drawing_bbox = True
            
    def on_mouse_up(self, x, y):
        """Handle mouse up events"""
        if (not self.is_point_in_image(x, y) or self.current_object_name is None or 
            self.annotation_mode != "bbox" or not self.is_drawing_bbox):
            return
            
        if self.bbox_start is not None:
            # Create bounding box
            start_x, start_y = self.bbox_start
            
            # Convert to image coordinates
            img_x1, img_y1 = self.canvas_to_image_coords(start_x, start_y)
            img_x2, img_y2 = self.canvas_to_image_coords(x, y)
            
            # Ensure proper ordering
            xmin, xmax = min(img_x1, img_x2), max(img_x1, img_x2)
            ymin, ymax = min(img_y1, img_y2), max(img_y1, img_y2)
            
            self.add_bounding_box(xmin, ymin, xmax, ymax)
            self.bbox_start = None
            self.is_drawing_bbox = False
            
    def on_mouse_move(self, x, y):
        """Handle mouse move events for bbox preview"""
        if (self.annotation_mode == "bbox" and self.is_drawing_bbox and 
            self.bbox_start is not None):
            
            # Use efficient redraw instead of full setup
            self.redraw_canvas_efficiently()
            
            # Draw preview rectangle
            start_x, start_y = self.bbox_start
            self.canvas.stroke_style = "red"
            self.canvas.line_width = 2
            self.canvas.set_line_dash([5, 5])
            self.canvas.stroke_rect(start_x, start_y, x - start_x, y - start_y)
            self.canvas.set_line_dash([])
            
    def draw_current_points(self):
        """Draw current working points"""
        for i, point in enumerate(self.points):
            x, y = point["x"], point["y"]
            canvas_x, canvas_y = self.image_to_canvas_coords(x, y)
            
            # Draw red circle for current points
            self.canvas.fill_style = "red"
            self.canvas.begin_path()
            self.canvas.arc(canvas_x, canvas_y, 5, 0, 2 * np.pi)
            self.canvas.fill()
            
            # Draw point number
            self.canvas.fill_style = "red"
            self.canvas.font = "12px Arial bold"
            self.canvas.fill_text(str(i + 1), canvas_x + 6, canvas_y - 6)
            
    def add_point(self, canvas_x, canvas_y):
        """Add a point annotation"""
        # Convert to image coordinates
        img_x, img_y = self.canvas_to_image_coords(canvas_x, canvas_y)
        
        self.points.append({"x": img_x, "y": img_y})
        self.total_points += 1
        
        # Efficiently redraw canvas to show the new point
        self.redraw_canvas_efficiently()
        
        print(f"Added point {len(self.points)}: ({img_x}, {img_y})")
        
    def add_bounding_box(self, xmin, ymin, xmax, ymax):
        """Add a bounding box annotation"""
        bbox_data = {
            "xmin": int(xmin),
            "ymin": int(ymin), 
            "xmax": int(xmax),
            "ymax": int(ymax)
        }
        
        # Store bbox for current object
        self.add_bounding_box_to_object(xmin, ymin, xmax, ymax)
        
        # Efficiently redraw canvas
        self.redraw_canvas_efficiently()
        
        print(f"Added bounding box: {bbox_data}")
        
    def set_object_name(self, object_name):
        """Set the current object being annotated"""
        self.current_object_name = object_name
        print(f"🎯 Set current object to: {object_name}")
            
    def set_annotation_mode(self, mode):
        """Set annotation mode: 'point' or 'bbox'"""
        self.annotation_mode = mode
        print(f"📝 Annotation mode set to: {mode}")
        
    def save_current_object(self):
        """Save the current object annotations"""
        if (self.current_object_name is None or self.current_pixel_key is None or 
            not self.points or self.annotations is None):
            print("❌ Cannot save: missing object name, pixel key, points, or data not loaded")
            return
            
        # Check if object already exists
        objects = self.annotations["pixel_keys"][self.current_pixel_key]["objects"]
        existing_obj = None
        for obj in objects:
            if obj["name"] == self.current_object_name:
                existing_obj = obj
                break
                
        if existing_obj is None:
            # Create new object
            new_object = {
                "name": self.current_object_name,
                "points": self.points.copy()
            }
            objects.append(new_object)
            print(f"✅ Created new object: {self.current_object_name}")
        else:
            # Update existing object points
            existing_obj["points"] = self.points.copy()
            print(f"✅ Updated points for object: {self.current_object_name}")
            
        # Clear current points and redraw
        self.points = []
        self.redraw_canvas_efficiently()
        
    def add_bounding_box_to_object(self, xmin, ymin, xmax, ymax):
        """Add bounding box to the current object"""
        if (self.current_object_name is None or self.current_pixel_key is None or 
            self.annotations is None):
            print("❌ Cannot add bbox: missing object name, pixel key, or data not loaded")
            return
            
        bbox_data = {
            "xmin": int(xmin),
            "ymin": int(ymin),
            "xmax": int(xmax), 
            "ymax": int(ymax)
        }
        
        # Find or create object
        objects = self.annotations["pixel_keys"][self.current_pixel_key]["objects"]
        existing_obj = None
        for obj in objects:
            if obj["name"] == self.current_object_name:
                existing_obj = obj
                break
                
        if existing_obj is None:
            new_object = {
                "name": self.current_object_name,
                "bounding_box": bbox_data,
                "points": []
            }
            objects.append(new_object)
            print(f"✅ Created new object with bbox: {self.current_object_name}")
        else:
            existing_obj["bounding_box"] = bbox_data
            print(f"✅ Updated bbox for object: {self.current_object_name}")
            
    def clear_current_annotations(self):
        """Clear current points and redraw"""
        self.points = []
        self.bbox_start = None
        self.is_drawing_bbox = False
        if self.canvas is not None:
            self.redraw_canvas_efficiently()
        print("🗑️ Cleared current annotations")
        
    def show_annotations(self):
        """Display current annotations"""
        if (self.annotations is None or 
            self.current_pixel_key not in self.annotations["pixel_keys"]):
            print("❌ No annotations for current image")
            return
            
        objects = self.annotations["pixel_keys"][self.current_pixel_key]["objects"]
        print(f"\n📊 Annotations for {self.current_pixel_key} (Task: {self.task_name}):")
        if not objects:
            print("  No objects annotated yet")
        else:
            for obj in objects:
                print(f"  🎯 Object: {obj['name']}")
                if "bounding_box" in obj:
                    print(f"    📦 Bounding box: {obj['bounding_box']}")
                if "points" in obj:
                    print(f"    📍 Points ({len(obj['points'])}): {obj['points']}")
            
    def save_annotations(self, json_filename="annotations.json"):
        """Save annotations to JSON and YAML files"""
        if self.annotations is None or self.task_name is None:
            print("❌ No annotations to save or task name not set")
            return

        self.task_dir = Path(REPO_PATH) / "coordinates" / self.task_name
        os.makedirs(self.task_dir, exist_ok=True)

        yaml_save_path = REPO_PATH / "point_policy" / "cfgs" / "suite" / "task" / "franka_env" / f"{self.task_name}.yaml"
        
        # Save JSON
        with open(self.task_dir / json_filename, 'w') as f:
            json.dump(self.annotations, f, indent=2)
        print(f"💾 Annotations saved to {self.task_dir / json_filename}")

        # Save images for both pixel keys
        for pixel_key in PIXEL_KEYS:
            try:
                img_bgr = self.data['observations'][0][pixel_key][0]
                image_filename = f"{pixel_key}.jpg"
                image_path = self.task_dir / image_filename
                cv2.imwrite(str(image_path), img_bgr)
                print(f"💾 Image saved to {image_path}")
            except Exception as e:
                print(f"❌ Error saving image for {pixel_key}: {e}")
                
        # Create YAML config
        unique_objects = set()
        for pixel_key_data in self.annotations["pixel_keys"].values():
            for obj in pixel_key_data["objects"]:
                unique_objects.add(obj["name"])
                
        yaml_config = {
            "defaults": ["_self_"],
            "task_name": self.task_name,
            "object_labels": ["objects"],   # TODO: see if this can be removed
            "num_object_points": self.total_points
        }
        
        with open(yaml_save_path, 'w') as f:
            yaml.dump(yaml_config, f, default_flow_style=False)
        print(f"💾 YAML config saved to {yaml_save_path}")
        
    def create_ui(self):
        """Create interactive UI widgets"""
        
        self.task_name_display = widgets.Text(
            value=self.task_name or 'Loading...',
            description='Task:',
            disabled=True,
            style={'description_width': '50px'},
            layout=widgets.Layout(width='300px')
        )
        
        # Camera and object controls
        self.pixel_key_dropdown = widgets.Dropdown(
            options=PIXEL_KEYS,
            value=PIXEL_KEYS[0],
            description='Camera:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='200px')
        )
        
        # Object input form
        self.object_input = widgets.Text(
            value='',
            placeholder='object name (e.g., gripper)',
            description='Object:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='250px')
        )
        
        self.set_object_btn = widgets.Button(
            description="Set", 
            button_style='info',
            layout=widgets.Layout(width='60px')
        )
        
        # Mode selection
        self.mode_radio = widgets.RadioButtons(
            options=['point', 'bbox'],
            value='point',
            description='Mode:',
            style={'description_width': '50px'},
            layout=widgets.Layout(width='130px')
        )
        
        # Action buttons
        self.save_object_btn = widgets.Button(description="Save Object", button_style='success')
        self.clear_btn = widgets.Button(description="Clear Current", button_style='warning')
        self.show_btn = widgets.Button(description="Show Annotations", button_style='')
        self.save_all_btn = widgets.Button(description="Save All", button_style='danger')
        
        # Button callbacks
        self.pixel_key_dropdown.observe(self._pixel_key_change_callback, names='value')
        self.set_object_btn.on_click(self._set_object_callback)
        self.save_object_btn.on_click(self._save_object_callback)
        self.clear_btn.on_click(self._clear_callback)
        self.show_btn.on_click(self._show_callback)
        self.save_all_btn.on_click(self._save_all_callback)
        self.mode_radio.observe(self._mode_change_callback, names='value')
        
        # Create canvas here so it can be included in layout
        if self.canvas is None:
            self.canvas = Canvas(width=self.canvas_width, height=self.canvas_height)
            self.canvas.on_mouse_down(self.on_mouse_down)
            self.canvas.on_mouse_up(self.on_mouse_up)
            self.canvas.on_mouse_move(self.on_mouse_move)
            # Setup canvas with current image
            self.setup_canvas()
        
        # Layout sections
        file_section = widgets.VBox([
            widgets.HTML("<h3>📁 Current Task</h3>"),
            self.task_name_display
        ])

        controls_section = widgets.VBox([
            widgets.HTML("<h3>🎯 Controls</h3>"),
            widgets.HBox([self.pixel_key_dropdown, self.mode_radio]),
            widgets.HBox([self.object_input, self.set_object_btn])
        ])
        
        actions_section = widgets.VBox([
            widgets.HTML("<h3>🔧 Actions</h3>"),
            widgets.VBox([
                self.save_object_btn,
                self.clear_btn,
                self.show_btn,
                self.save_all_btn
            ])
        ])
        
        # Canvas section with title
        canvas_title = widgets.HTML(f"<h3>🖼️ {self.update_canvas_title()}</h3>")
        canvas_section = widgets.VBox([canvas_title, self.canvas])
        
        # Layout - put controls and canvas side by side
        controls_column = widgets.VBox([file_section, controls_section, actions_section])
        
        # Put controls and canvas side by side
        ui = widgets.HBox([controls_column, canvas_section])
        return ui
        
    def _pixel_key_change_callback(self, change):
        """Handle pixel key dropdown change"""
        self.switch_pixel_key(change['new'])
            
    def _set_object_callback(self, b):
        """Handle set object button click"""
        object_name = self.object_input.value.strip()
        if object_name:
            self.set_object_name(object_name)
        else:
            print("❌ Please enter an object name")
        
    def _save_object_callback(self, b):
        self.save_current_object()
        
    def _clear_callback(self, b):
        self.clear_current_annotations()
        
    def _show_callback(self, b):
        self.show_annotations()
        
    def _save_all_callback(self, b):
        self.save_annotations()
        
    def _mode_change_callback(self, change):
        self.set_annotation_mode(change['new'])

# Usage function
def demo_annotation_tool():
    # Create annotation tool
    tool = ImageAnnotationTool()
    
    # Create and display UI
    ui = tool.create_ui()
    display(ui)
    
    return tool

# Run the demo
annotation_tool = demo_annotation_tool()

# TODO: change the json serialization 

🔄 Switched to pixels1
✅ Auto-loaded: bowl.pkl
📝 Task name: bowl


🔄 Switched to pixels2
🔄 Switched to pixels1
🎯 Set current object to: bowl
Added point 1: (90, 203)
Added point 2: (81, 224)
Added point 3: (114, 209)
Added point 4: (105, 229)
📝 Annotation mode set to: bbox
✅ Created new object with bbox: bowl
Added bounding box: {'xmin': 65, 'ymin': 188, 'xmax': 127, 'ymax': 240}
✅ Updated points for object: bowl
🎯 Set current object to: lemon
📝 Annotation mode set to: point


{'task_name': 'bowl', 'pixel_keys': {'pixels1': {'image_path': 'pixels1.jpg', 'objects': [{'name': 'bowl', 'bounding_box': {'xmin': 65, 'ymin': 188, 'xmax': 127, 'ymax': 240}, 'points': [{'x': 90, 'y': 203}, {'x': 81, 'y': 224}, {'x': 114, 'y': 209}, {'x': 105, 'y': 229}]}]}, 'pixels2': {'image_path': 'pixels2.jpg', 'objects': []}}}
